# Z2005 — Week 05: Advanced Sorting

A self-study notebook on merge sort and quicksort — the divide-and-conquer
paradigm, partition schemes, worst-case behavior and pivot selection,
stability, and how to choose a sorting algorithm for a real workload.


## Learning Objectives

By the end of this notebook you will be able to:

- Implement merge sort (recursive and iterative/bottom-up) and explain why it is always O(n log n) and always stable.
- Implement quicksort with the Lomuto partition scheme and trace how a pivot splits the array.
- Explain quicksort's worst case (O(n^2), triggered by consistently bad pivot choices) and describe pivot-selection strategies that avoid it in practice.
- Compare merge sort and quicksort on time complexity, space complexity, and stability, and justify why each is preferred in different situations.
- Use `timeit` to empirically compare merge sort and quicksort against each other and against the simple sorts from Week 4.
- Choose an appropriate sorting algorithm for a given dataset (size, memory constraints, existing order, key type).


## How to use this notebook

Run the cells top to bottom. Markdown cells explain a concept; the code cell
right after it is a fully worked, heavily commented example. Cells marked
`# TODO` in **Exercises** are for you to complete — replace
`raise NotImplementedError` with your own code. Each exercise is followed by
an assert-based **Self-Check** cell: it raises `AssertionError` if your code
is wrong, and prints a success message if it is right. Solutions are
collected at the end — attempt the exercises first.


## 1. Divide and conquer

Merge sort and quicksort both follow the **divide-and-conquer** paradigm:
break a problem into smaller subproblems of the same shape, solve each
recursively, then combine the sub-results. The pattern is always:

1. **Divide** — split the input into pieces.
2. **Conquer** — recursively solve each piece (the recursion bottoms out at
   a trivially small piece, e.g. length 0 or 1, which is "already sorted").
3. **Combine** — merge/assemble the solved pieces into the full solution.

Merge sort puts almost all of its work in the *combine* step (the merge);
quicksort puts almost all of its work in the *divide* step (the partition).
That single difference explains most of the contrasts between them: merge
sort needs O(n) extra space for merging, while quicksort's partition can be
done in place.


In [ ]:
def merge(left, right):
    """Combine two already-sorted lists into one sorted list. O(len(left)+len(right))."""
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:      # <= (not <) is what makes merge sort stable:
            result.append(left[i]);  i += 1   # a left-side tie wins, preserving original order
        else:
            result.append(right[j]); j += 1
    result.extend(left[i:])          # copy any leftover tail (at most one side has one)
    result.extend(right[j:])
    return result


def merge_sort(arr):
    if len(arr) <= 1:                # base case: 0 or 1 elements is trivially sorted
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])     # divide + conquer left half
    right = merge_sort(arr[mid:])    # divide + conquer right half
    return merge(left, right)        # combine


assert merge([2, 5, 8], [1, 3, 9]) == [1, 2, 3, 5, 8, 9]
assert merge_sort([8, 3, 5, 1, 9, 2, 7, 4]) == [1, 2, 3, 4, 5, 7, 8, 9]
assert merge_sort([]) == []
assert merge_sort([1]) == [1]
print("Merge sort checks passed")


## 2. Merge sort's complexity, and the bottom-up variant

**Time:** each of the `log2(n)` levels of recursion does O(n) work in total
across all merges at that level, giving O(n log n) in the best, average, and
worst case alike — merge sort has no bad input, unlike quicksort.
**Space:** O(n) auxiliary space for the merge buffers (plus O(log n) stack
space for recursion) — this is the real cost of putting all the work in
`merge`.
**Stability:** yes, guaranteed by the `<=` comparison in `merge` shown above.

An equivalent **bottom-up (iterative)** version avoids recursion entirely:
instead of splitting top-down, it starts by merging adjacent pairs of
single elements, then merges adjacent pairs of 2-element runs, then 4, and
so on, doubling the merged-run width each pass.


In [ ]:
def merge_sort_iterative(arr):
    arr = arr[:]
    n = len(arr)
    width = 1
    while width < n:                              # doubling: 1, 2, 4, 8, ... -> log2(n) passes
        for i in range(0, n, 2 * width):
            left = arr[i:i + width]
            right = arr[i + width:i + 2 * width]  # may be shorter than `width` on the last pair
            arr[i:i + len(left) + len(right)] = merge(left, right)
        width *= 2
    return arr


assert merge_sort_iterative([5, 2, 8, 1]) == [1, 2, 5, 8]
assert merge_sort_iterative([8, 3, 5, 1, 9, 2, 7, 4]) == [1, 2, 3, 4, 5, 7, 8, 9]
assert merge_sort_iterative([]) == []
assert merge_sort_iterative([7, 7, 3]) == [3, 7, 7]
print("Iterative merge sort checks passed")


## 3. Quicksort with the Lomuto partition scheme

Quicksort picks a **pivot**, then **partitions** the array so that everything
less than the pivot ends up to its left and everything greater ends up to
its right — the pivot is now in its final sorted position. It then recurses
on the two sides independently. The Lomuto scheme (below) always picks the
last element as the pivot and uses a single index `i` to track the boundary
of "elements confirmed less than the pivot so far."

Unlike merge sort, all of quicksort's work happens in `partition`, and it
happens **in place** — no auxiliary array is needed, which is quicksort's
main practical advantage over merge sort.


In [ ]:
def partition(arr, low, high):
    """Lomuto partition: pivot = arr[high]. Rearranges arr[low:high+1] in
    place so everything < pivot comes before it and everything >= pivot
    comes after; returns the pivot's final index."""
    pivot = arr[high]
    i = low - 1                       # i tracks the last index known to be < pivot
    for j in range(low, high):
        if arr[j] < pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]   # extend the "less than pivot" region by one
    arr[i + 1], arr[high] = arr[high], arr[i + 1]   # place pivot right after that region
    return i + 1


def quicksort(arr, low=0, high=None):
    if high is None:
        high = len(arr) - 1
    if low < high:                    # base case: 0 or 1 elements needs no work
        pivot_index = partition(arr, low, high)
        quicksort(arr, low, pivot_index - 1)    # recurse left of the pivot
        quicksort(arr, pivot_index + 1, high)   # recurse right of the pivot
    return arr


test_arr = [8, 3, 5, 1, 9, 2]
assert partition(test_arr, 0, 5) == 1
assert test_arr == [1, 2, 5, 8, 9, 3]   # pivot (2) now sits at its final index 1
assert quicksort([8, 3, 5, 1, 9, 2, 7, 4]) == [1, 2, 3, 4, 5, 7, 8, 9]
assert quicksort([]) == []
assert quicksort([1]) == [1]
print("Quicksort checks passed")


## 4. Quicksort's worst case, and pivot selection strategies

With Lomuto's "always pick the last element" rule, an **already-sorted** (or
reverse-sorted) array is quicksort's worst case: every partition splits the
array into a piece of size 0 and a piece of size n-1, giving n levels of
recursion instead of log n, for O(n^2) total time — and O(n) recursion depth,
which can even overflow the call stack on large inputs.

**Fixes, in increasing order of robustness:**

- **Randomized pivot:** swap a uniformly random element into the pivot
  position before partitioning. This makes the worst case *astronomically*
  unlikely for any fixed input, at the cost of one `random.randint` call per
  partition.
- **Median-of-three:** pick the median of the first, middle, and last
  elements as the pivot — cheap, and defeats the common "already sorted"
  and "reverse sorted" adversarial cases without any randomness.
- **Introsort (what production sorts actually do):** run quicksort but
  switch to heapsort if the recursion depth exceeds a bound like
  `2 * log2(n)`, guaranteeing O(n log n) worst case no matter what an
  adversary does.


In [ ]:
import random
import sys

def partition_random(arr, low, high):
    """Lomuto partition with a uniformly random pivot, swapped into the
    'last element' slot Lomuto expects before partitioning as usual."""
    r = random.randint(low, high)
    arr[r], arr[high] = arr[high], arr[r]
    return partition(arr, low, high)


def quicksort_random(arr, low=0, high=None, depth_tracker=None):
    if high is None:
        high = len(arr) - 1
    if depth_tracker is None:
        depth_tracker = [0, 0]        # [current_depth, max_depth_seen]
    if low < high:
        depth_tracker[0] += 1
        depth_tracker[1] = max(depth_tracker[1], depth_tracker[0])
        pivot_index = partition_random(arr, low, high)
        quicksort_random(arr, low, pivot_index - 1, depth_tracker)
        quicksort_random(arr, pivot_index + 1, high, depth_tracker)
        depth_tracker[0] -= 1
    return arr


sys.setrecursionlimit(3000)

# Demonstrate the worst case directly: plain (deterministic) quicksort on
# already-sorted input recurses to depth ~n.
sorted_input_small = list(range(300))
sys.setrecursionlimit(2000)
depth_deterministic = [0, 0]

def quicksort_depth_tracked(arr, low=0, high=None, depth_tracker=None):
    if high is None:
        high = len(arr) - 1
    if depth_tracker is None:
        depth_tracker = [0, 0]
    if low < high:
        depth_tracker[0] += 1
        depth_tracker[1] = max(depth_tracker[1], depth_tracker[0])
        pivot_index = partition(arr, low, high)
        quicksort_depth_tracked(arr, low, pivot_index - 1, depth_tracker)
        quicksort_depth_tracked(arr, pivot_index + 1, high, depth_tracker)
        depth_tracker[0] -= 1
    return arr

quicksort_depth_tracked(sorted_input_small[:], depth_tracker=depth_deterministic)

depth_random = [0, 0]
quicksort_random(sorted_input_small[:], depth_tracker=depth_random)

print(f"already-sorted input, n=300:")
print(f"  deterministic (last-element pivot) max recursion depth: {depth_deterministic[1]}")
print(f"  randomized pivot max recursion depth:                   {depth_random[1]}")
assert depth_deterministic[1] == 299   # worst case confirmed: depth == n-1 on sorted input
assert depth_random[1] < depth_deterministic[1]   # randomization measurably avoids it

sorted_input_500 = list(range(500))
result = quicksort_random(sorted_input_500[:], depth_tracker=[0, 0])
assert result == sorted(sorted_input_500)
print("Randomized quicksort correctness and worst-case-avoidance checks passed")


## 5. Stability, and merge sort vs. quicksort

Merge sort is **stable** (Section 2). Quicksort, as implemented with Lomuto
partitioning, is **not stable** in general: swapping elements during
partitioning can reorder equal keys relative to each other, and there is no
cheap fix that preserves quicksort's in-place, O(log n)-space advantage
(making it stable requires effectively the same O(n) auxiliary space that
merge sort already uses, defeating the point).

| Property | Merge sort | Quicksort |
|---|---|---|
| Best case | O(n log n) | O(n log n) |
| Average case | O(n log n) | O(n log n) |
| Worst case | O(n log n) | O(n^2) (mitigated by randomization/median-of-three) |
| Extra space | O(n) | O(log n) expected (recursion stack only) |
| Stable | Yes | No |
| In-place | No | Yes |
| Typical practical speed | Good | Usually faster in practice (better cache locality, smaller constants) |


In [ ]:
def quicksort_tagged_partition(pairs, low, high):
    pivot = pairs[high][0]
    i = low - 1
    for j in range(low, high):
        if pairs[j][0] < pivot:
            i += 1
            pairs[i], pairs[j] = pairs[j], pairs[i]
    pairs[i + 1], pairs[high] = pairs[high], pairs[i + 1]
    return i + 1


def quicksort_tagged(pairs, low=0, high=None):
    if high is None:
        high = len(pairs) - 1
    if low < high:
        p = quicksort_tagged_partition(pairs, low, high)
        quicksort_tagged(pairs, low, p - 1)
        quicksort_tagged(pairs, p + 1, high)
    return pairs


tagged = [(3, "a"), (3, "b"), (1, "c")]
result = quicksort_tagged(tagged[:])
# Only assert the KEYS are sorted -- do not assume any particular tag order,
# because Lomuto partitioning is not guaranteed to preserve it.
assert [k for k, _ in result] == [1, 3, 3]
print("Quicksort produces correct key order; tag order is not guaranteed (unstable), by design")


## 6. Empirical timing: advanced sorts vs. simple sorts

Let's measure, not assume. We re-implement the Week 4 simple sorts here
(bubble, selection, insertion) and time all four algorithms plus Python's
built-in `sorted()` (Timsort) on the same random data, using `timeit`.
Expect the O(n^2) sorts to fall badly behind as n grows, and Python's
built-in `sorted()` to win outright — it is a highly tuned C implementation
of a hybrid merge/insertion sort (Timsort).


In [ ]:
import timeit

def bubble_sort(arr):
    arr = arr[:]
    n = len(arr)
    for i in range(n - 1):
        swapped = False
        for j in range(n - 1 - i):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swapped = True
        if not swapped:
            break
    return arr

def selection_sort(arr):
    arr = arr[:]
    n = len(arr)
    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):
            if arr[j] < arr[min_idx]:
                min_idx = j
        arr[i], arr[min_idx] = arr[min_idx], arr[i]
    return arr

def insertion_sort(arr):
    arr = arr[:]
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr


random.seed(11)
n = 800
data = [random.randint(0, 100000) for _ in range(n)]

timings = {
    "bubble_sort":      timeit.timeit(lambda: bubble_sort(data), number=2),
    "selection_sort":   timeit.timeit(lambda: selection_sort(data), number=2),
    "insertion_sort":   timeit.timeit(lambda: insertion_sort(data), number=2),
    "merge_sort":       timeit.timeit(lambda: merge_sort(data), number=2),
    "quicksort":        timeit.timeit(lambda: quicksort(data[:]), number=2),
    "sorted (Timsort)": timeit.timeit(lambda: sorted(data), number=2),
}

print(f"Sorting n={n} random integers, best of runs shown, seconds for 2 repeats:")
for name, t in sorted(timings.items(), key=lambda kv: kv[1]):
    print(f"  {name:18s} {t:.4f}s")

# Empirical sanity checks derived from what we just measured, not assumed:
assert timings["sorted (Timsort)"] < timings["bubble_sort"]
assert timings["merge_sort"] < timings["bubble_sort"]
assert timings["quicksort"] < timings["selection_sort"]
print("\nConfirmed: built-in Timsort and both O(n log n) sorts measurably beat the O(n^2) sorts")


## Exercises

Try each one yourself before checking the Solutions section at the end.


**Exercise 1 — `merge_k_sorted_lists`.** Given a list of `k` already-sorted
lists, merge them all into one fully sorted list. Do it by repeatedly
applying the two-list `merge` function from Section 1 (pairwise, or via
`functools.reduce`) — do not just concatenate and call `sorted()`.

Example: `[[1, 4, 7], [2, 3], [0, 5, 6]]` -> `[0, 1, 2, 3, 4, 5, 6, 7]`.


In [ ]:
def merge_k_sorted_lists(lists):
    """Merge a list of k already-sorted lists into one sorted list, using
    the pairwise `merge` function.

    Args:
        lists: a list of k lists, each individually sorted ascending. k may
            be 0 (return []).
    Returns:
        A single sorted list containing every element from every input list.
    """
    # TODO: implement this. Hint: fold `merge` over the lists one at a time
    # (e.g. start with result = [], then result = merge(result, lst) for
    # each lst in lists), or use functools.reduce(merge, lists, []).
    raise NotImplementedError


In [ ]:
# Self-Check: Exercise 1
assert merge_k_sorted_lists([[1, 4, 7], [2, 3], [0, 5, 6]]) == [0, 1, 2, 3, 4, 5, 6, 7]
assert merge_k_sorted_lists([]) == []
assert merge_k_sorted_lists([[1, 2, 3]]) == [1, 2, 3]
assert merge_k_sorted_lists([[], [1], []]) == [1]
print("Exercise 1 passed")


**Exercise 2 — `quicksort_hoare`.** Implement quicksort using the **Hoare
partition scheme** instead of Lomuto. Hoare's scheme uses two indices
starting at both ends and moving inward, and (unlike Lomuto) does not
guarantee the pivot itself ends up at the returned index — instead, the
returned index `p` guarantees `arr[low..p]` are all `<= pivot` and
`arr[p+1..high]` are all `>= pivot`, so the recursive calls are
`quicksort_hoare(arr, low, p)` and `quicksort_hoare(arr, p+1, high)` (note:
not `p-1`, unlike Lomuto). Hoare's scheme does about 3x fewer swaps than
Lomuto on average.

Example: `[8, 3, 5, 1, 9, 2, 7, 4]` -> `[1, 2, 3, 4, 5, 7, 8, 9]`.


In [ ]:
def hoare_partition(arr, low, high):
    """Hoare partition using arr[low] as the pivot. Returns index p such
    that arr[low..p] <= pivot <= arr[p+1..high] (not necessarily the
    pivot's own final index).

    Args:
        arr: list to partition in place.
        low, high: inclusive bounds of the region to partition.
    Returns:
        The partition index p, for use as quicksort_hoare(arr, low, p) and
        quicksort_hoare(arr, p+1, high).
    """
    # TODO: implement this. Hint: pivot = arr[low]; i = low - 1; j = high + 1;
    # loop: increment i while arr[i] < pivot; decrement j while arr[j] > pivot;
    # if i >= j return j; otherwise swap arr[i] and arr[j] and repeat.
    raise NotImplementedError


def quicksort_hoare(arr, low=0, high=None):
    """Sort arr[low..high] in place using Hoare-partition quicksort."""
    if high is None:
        high = len(arr) - 1
    if low < high:
        p = hoare_partition(arr, low, high)
        quicksort_hoare(arr, low, p)         # NOTE: p, not p - 1 (Hoare, unlike Lomuto)
        quicksort_hoare(arr, p + 1, high)
    return arr


In [ ]:
# Self-Check: Exercise 2
assert quicksort_hoare([8, 3, 5, 1, 9, 2, 7, 4]) == [1, 2, 3, 4, 5, 7, 8, 9]
assert quicksort_hoare([]) == []
assert quicksort_hoare([1]) == [1]
assert quicksort_hoare([2, 1]) == [1, 2]

random.seed(4)
big = [random.randint(0, 1000) for _ in range(200)]
assert quicksort_hoare(big[:]) == sorted(big)
print("Exercise 2 passed")


**Exercise 3 — `kth_smallest_quickselect`.** Using the Lomuto `partition`
function from Section 3, implement **quickselect**: find the k-th smallest
element (0-indexed, so k=0 is the minimum) of an unsorted list in expected
O(n) time, *without* fully sorting the list. The key idea: after one
partition step, if the pivot's final index equals k you are done; otherwise
recurse into only the one side that could contain the k-th smallest.

Example: `kth_smallest_quickselect([7, 2, 9, 4, 1], 2)` -> `4` (the 3rd
smallest, i.e. index 2, of `[1, 2, 4, 7, 9]`).


In [ ]:
def kth_smallest_quickselect(arr, k):
    """Return the k-th smallest element (0-indexed) of arr in expected
    O(n) time using quickselect, built on the Lomuto `partition` function.
    Does not need to fully sort arr.

    Args:
        arr: list of comparable elements (0 <= k < len(arr)).
        k: the 0-indexed rank to find (k=0 is the minimum).
    Returns:
        The k-th smallest value in arr.
    """
    # TODO: implement this. Hint: work on a copy of arr. Use partition(copy,
    # low, high) as in Section 3. If the returned pivot index equals k,
    # return copy[k]. If it's greater than k, recurse into the left side
    # (low, pivot_index - 1); otherwise recurse into the right side
    # (pivot_index + 1, high). This only recurses into ONE side, unlike
    # full quicksort, which is what gives it expected O(n) instead of O(n log n).
    raise NotImplementedError


In [ ]:
# Self-Check: Exercise 3
assert kth_smallest_quickselect([7, 2, 9, 4, 1], 2) == 4
assert kth_smallest_quickselect([7, 2, 9, 4, 1], 0) == 1     # minimum
assert kth_smallest_quickselect([7, 2, 9, 4, 1], 4) == 9     # maximum
assert kth_smallest_quickselect([5], 0) == 5

random.seed(6)
sample = [random.randint(0, 500) for _ in range(100)]
expected = sorted(sample)
for k in (0, 1, 50, 98, 99):
    assert kth_smallest_quickselect(sample, k) == expected[k]
print("Exercise 3 passed")


**Exercise 4 (harder) — `choose_sort_algorithm`.** Write a function that,
given a small description of a sorting workload, returns a string naming
the *best-justified* choice among `"insertion_sort"`, `"counting_sort"`,
`"merge_sort"`, or `"quicksort"`, following the decision guidance from
Section 5/6 of this notebook and Week 4:

- If `n <= 32`: `"insertion_sort"` (constant-factor win on small inputs,
  matching why production sorts fall back to it).
- Else if `key_range is not None` and `key_range <= 4 * n`: `"counting_sort"`
  (bounded, reasonably small integer key range relative to n).
- Else if `stability_required` is `True`: `"merge_sort"` (the only stable
  O(n log n) option among these four).
- Else: `"quicksort"` (best typical practical speed when stability is not
  required and the key range is not favorably bounded).

Example: `choose_sort_algorithm(n=10, key_range=None, stability_required=False)`
-> `"insertion_sort"` (n is small, so this rule fires first regardless of
the other arguments).


In [ ]:
def choose_sort_algorithm(n, key_range=None, stability_required=False):
    """Recommend a sort algorithm name for a workload, per the decision
    rules in this notebook's Section 5/6 discussion.

    Args:
        n: number of elements to sort.
        key_range: k such that all keys lie in [0, k], or None if keys are
            not bounded small integers.
        stability_required: True if equal-keyed elements must keep their
            relative input order.
    Returns:
        One of "insertion_sort", "counting_sort", "merge_sort", "quicksort".
    """
    # TODO: implement this using the four rules described above, checked
    # in the given order (small-n check first, then key-range check, then
    # stability check, else quicksort).
    raise NotImplementedError


In [ ]:
# Self-Check: Exercise 4
assert choose_sort_algorithm(n=10, key_range=None, stability_required=False) == "insertion_sort"
assert choose_sort_algorithm(n=500, key_range=100, stability_required=False) == "counting_sort"
assert choose_sort_algorithm(n=10000, key_range=None, stability_required=True) == "merge_sort"
assert choose_sort_algorithm(n=10000, key_range=None, stability_required=False) == "quicksort"
# key_range present but too large relative to n should NOT trigger counting sort
assert choose_sort_algorithm(n=100, key_range=10_000_000, stability_required=False) == "quicksort"
print("Exercise 4 passed")


## Quiz

**Q1.** Merge sort is O(n log n) in the worst case; quicksort (with a fixed,
non-randomized pivot rule) is O(n^2) in the worst case. Given this, why is
quicksort still often preferred in practice over merge sort?

<details><summary>Show answer</summary>
Quicksort's worst case is avoidable in practice (randomized pivots or
median-of-three make it vanishingly unlikely to be triggered), and when it
does not hit that worst case, quicksort typically runs faster than merge
sort in real measurements because it sorts in place (no O(n) auxiliary
array to allocate and copy into, unlike merge sort) and tends to have better
cache locality. Section 6's timing experiment shows both beating the simple
O(n^2) sorts, and quicksort is usually the faster of the two in practice
despite the theoretically worse worst case.
</details>

**Q2.** In the Lomuto partition scheme, what does the index `i` represent
at any point during the loop, and why must the pivot be swapped with
`arr[i+1]` (not `arr[i]`) at the end?

<details><summary>Show answer</summary>
`i` is the index of the last element confirmed to be less than the pivot
(it starts at `low - 1`, meaning "no elements confirmed yet"). After the
loop, everything in `arr[low..i]` is `< pivot`, and everything in
`arr[i+1..high-1]` is `>= pivot`. The pivot itself (currently at `arr[high]`)
belongs right after the "less than" region, i.e. at index `i+1` — swapping
it with `arr[i]` instead would place it one slot too early, inside the
region that is supposed to be strictly less than it.
</details>

**Q3.** A colleague claims: "Randomizing the pivot in quicksort guarantees
O(n log n) worst-case time." Is this claim accurate? What is the correct,
more careful statement?

<details><summary>Show answer</summary>
Not accurate. Randomization does not change the *worst-case* time
complexity, which is still O(n^2) — it is always mathematically possible
for the random choices to happen to be bad every time, however astronomically
unlikely. What randomization actually guarantees is that the *expected*
(average-case) running time is O(n log n) for *any* input, because no
specific input (like an already-sorted array) can be constructed in advance
to reliably trigger bad behavior against a pivot chosen at random each time.
</details>

**Q4.** For sorting 50 million fixed-size log records by timestamp on a
memory-constrained server, which of merge sort or quicksort is the more
appropriate default, and why?

<details><summary>Show answer</summary>
Quicksort (with randomized or median-of-three pivot selection), because it
sorts in place with only O(log n) auxiliary stack space, whereas merge sort
needs O(n) auxiliary space for its merge buffers — at 50 million records,
that auxiliary array could itself become a memory problem on a
memory-constrained machine. This matches Week 5 Lecture 3's "50 million log
entries, memory-constrained server" scenario. (If strict stability were also
required, merge sort's O(n) space cost might have to be accepted anyway,
since quicksort cannot offer stability at O(log n) space.)
</details>


## Solutions (try the exercises yourself first!)


In [ ]:
# Solution: Exercise 1
import functools

def merge_k_sorted_lists(lists):
    return functools.reduce(merge, lists, [])

assert merge_k_sorted_lists([[1, 4, 7], [2, 3], [0, 5, 6]]) == [0, 1, 2, 3, 4, 5, 6, 7]
assert merge_k_sorted_lists([]) == []
print("Exercise 1 solution verified")


In [ ]:
# Solution: Exercise 2
def hoare_partition(arr, low, high):
    pivot = arr[low]
    i, j = low - 1, high + 1
    while True:
        i += 1
        while arr[i] < pivot:
            i += 1
        j -= 1
        while arr[j] > pivot:
            j -= 1
        if i >= j:
            return j
        arr[i], arr[j] = arr[j], arr[i]

def quicksort_hoare(arr, low=0, high=None):
    if high is None:
        high = len(arr) - 1
    if low < high:
        p = hoare_partition(arr, low, high)
        quicksort_hoare(arr, low, p)
        quicksort_hoare(arr, p + 1, high)
    return arr

assert quicksort_hoare([8, 3, 5, 1, 9, 2, 7, 4]) == [1, 2, 3, 4, 5, 7, 8, 9]
print("Exercise 2 solution verified")


In [ ]:
# Solution: Exercise 3
def kth_smallest_quickselect(arr, k):
    copy = arr[:]
    low, high = 0, len(copy) - 1
    while True:
        pivot_index = partition(copy, low, high)
        if pivot_index == k:
            return copy[k]
        elif pivot_index > k:
            high = pivot_index - 1
        else:
            low = pivot_index + 1

assert kth_smallest_quickselect([7, 2, 9, 4, 1], 2) == 4
assert kth_smallest_quickselect([7, 2, 9, 4, 1], 0) == 1
print("Exercise 3 solution verified")


In [ ]:
# Solution: Exercise 4
def choose_sort_algorithm(n, key_range=None, stability_required=False):
    if n <= 32:
        return "insertion_sort"
    if key_range is not None and key_range <= 4 * n:
        return "counting_sort"
    if stability_required:
        return "merge_sort"
    return "quicksort"

assert choose_sort_algorithm(n=10, key_range=None, stability_required=False) == "insertion_sort"
assert choose_sort_algorithm(n=500, key_range=100, stability_required=False) == "counting_sort"
assert choose_sort_algorithm(n=10000, key_range=None, stability_required=True) == "merge_sort"
assert choose_sort_algorithm(n=10000, key_range=None, stability_required=False) == "quicksort"
print("Exercise 4 solution verified")
